In [2]:
import pandas as pd

# 1. Load your uploaded data
df = pd.read_csv(r'..\data\train.csv')

# 2. Define the correct target labels for these specific columns
labels_map = {
    'Row ID': 'ID',
    'Order ID': 'ID',
    'Order Date': 'Date',
    'Ship Date': 'Date',
    'Ship Mode': 'Categorical',
    'Customer ID': 'ID',
    'Customer Name': 'Text',
    'Segment': 'Categorical',
    'Country': 'Location',
    'City': 'Location',
    'State': 'Location',
    'Postal Code': 'Location',
    'Region': 'Categorical',
    'Product ID': 'ID',
    'Category': 'Categorical',
    'Sub-Category': 'Categorical',
    'Product Name': 'Text',
    'Sales': 'Numeric'
}

training_rows = []

# 3. Loop through columns to generate the training format
for col in df.columns:
    # Get up to 3 non-null sample values as a single string
    samples = df[col].dropna().astype(str).unique()[:3]
    sample_str = ", ".join(samples)
    
    # Get the exact data type
    dtype_str = str(df[col].dtype)
    
    # Get the assigned label
    label = labels_map.get(col, "Unknown")
    
    training_rows.append({
        "column_name": col,
        "sample_values": sample_str,
        "dtype": dtype_str,
        "label": label
    })

# 4. Create the training dataframe
train_metadata_df = pd.DataFrame(training_rows)

# 5. Save it to CSV for the next step
train_metadata_df.to_csv('your_training_data.csv', index=False)

print("Training data created successfully!")
display(train_metadata_df.head())

Training data created successfully!


,column_name,sample_values,dtype,label
0,Row ID,"1, 2, 3",int64,ID
1,Order ID,"CA-2017-152156, CA-2017-138688, US-2016-108966",object,ID
2,Order Date,"08/11/2017, 12/06/2017, 11/10/2016",object,Date
3,Ship Date,"11/11/2017, 16/06/2017, 18/10/2016",object,Date
4,Ship Mode,"Second Class, Standard Class, First Class",object,Categorical


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load the training data created in Cell 1
try:
    df = pd.read_csv('your_training_data.csv')
    df.fillna('', inplace=True) # Handle any missing values safely
except FileNotFoundError:
    print("Error: 'your_training_data.csv' not found. Please ensure Cell 1 ran successfully.")
    raise

# 2. Separate Features (X) and Target Label (y)
X = df[['column_name', 'sample_values', 'dtype']]
y = df['label']

# 3. Encode the target labels into numbers (e.g., 'ID' -> 0, 'Date' -> 1)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 4. Build the Preprocessing Column Transformer
# This converts our text and categorical columns into a format the math model understands
preprocessor = ColumnTransformer(
    transformers=[
        ('col_name_tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=100), 'column_name'),
        ('samples_tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=100), 'sample_values'),
        ('dtype_ohe', OneHotEncoder(handle_unknown='ignore'), ['dtype'])
    ])

# 5. Create the full training Pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 6. Train the model!
print("Training model...")
model_pipeline.fit(X, y_encoded)
print("Model training complete!")

# 7. Save the model and label encoder
# These will be saved in the same directory your notebook is running in.
joblib.dump(model_pipeline, 'column_mapper_model.pkl')
joblib.dump(label_encoder, 'column_mapper_label_encoder.pkl')

print("\nSuccess! Files saved:")
print(" - column_mapper_model.pkl")
print(" - column_mapper_label_encoder.pkl")

# --- Optional: Quick Sanity Check ---
print("\n--- Testing the model on new unseen columns ---")
test_data = pd.DataFrame({
    'column_name': ['Client_ID', 'Purchase_Date', 'Shipping_City', 'Total_Cost'],
    'sample_values': ['C-991, C-002', '2023-10-01, 2023-10-05', 'New York, London', '150.50, 20.00'],
    'dtype': ['object', 'object', 'object', 'float64']
})

# Make predictions
preds = model_pipeline.predict(test_data)
# Convert numerical predictions back to readable text labels
decoded_preds = label_encoder.inverse_transform(preds)

for col, pred in zip(test_data['column_name'], decoded_preds):
    print(f"Column '{col}' predicted as -> {pred}")

Training model...
Model training complete!

Success! Files saved:
 - column_mapper_model.pkl
 - column_mapper_label_encoder.pkl

--- Testing the model on new unseen columns ---
Column 'Client_ID' predicted as -> Location
Column 'Purchase_Date' predicted as -> Location
Column 'Shipping_City' predicted as -> Location
Column 'Total_Cost' predicted as -> Location


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load the training data with the required 'label' column
# Reuse the dataframe created in the earlier notebook cell if available.
try:
    df = train_metadata_df.copy()
except NameError:
    training_file_path = 'your_training_data.csv'
    try:
        df = pd.read_csv(training_file_path)
    except FileNotFoundError:
        print(f"Error: '{training_file_path}' not found. Please run the earlier cell that creates it.")
        raise

df.fillna('', inplace=True)

# 2. Enforce Canonical Labels Only

canonical_mapping = {
    'ID': 'id',
    'Date': 'datetime',
    'Categorical': 'categorical',
    'Location': 'categorical',
    'Numeric': 'numeric',
    'Text': 'text',
    'Unknown': 'text'
}

print("Enforcing canonical labels...")
# Standardize the labels based on the mapping, defaulting to 'text' if it doesn't match
df['canonical_label'] = df['label'].map(lambda x: canonical_mapping.get(x, str(x).lower()))

# Separate Features (X) and Target (y)
X = df[['column_name', 'sample_values', 'dtype']]
y = df['canonical_label']

# 3. Encode the target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Training on the following canonical classes: {label_encoder.classes_}")

# 4. Build Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('col_name_tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=100), 'column_name'),
        ('samples_tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=100), 'sample_values'),
        ('dtype_ohe', OneHotEncoder(handle_unknown='ignore'), ['dtype'])
    ])

# 5. Create Pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, random_state=42)) # Increased trees for a larger dataset
])

# 6. Train the model
print(f"Retraining model on {len(df)} rows...")
model_pipeline.fit(X, y_encoded)
print("Model retraining complete!")

# 7. Overwrite the files

joblib.dump(model_pipeline, 'column_mapper_model.pkl')
joblib.dump(label_encoder, 'column_mapper_label_encoder.pkl')

print("\nSuccess! Files overwritten locally:")
print(" - column_mapper_model.pkl")
print(" - column_mapper_label_encoder.pkl")

Enforcing canonical labels...
Training on the following canonical classes: ['categorical' 'datetime' 'id' 'numeric' 'text']
Retraining model on 18 rows...
Model retraining complete!

Success! Files overwritten locally:
 - column_mapper_model.pkl
 - column_mapper_label_encoder.pkl


In [ ]:
import os
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

r
def build_feature_text(frame):
    col_name = frame['column_name'].astype(str)
    sample_values = frame['sample_values'].fillna('').astype(str)
    dtypes = frame['dtype'].fillna('unknown').astype(str)
    return 'name:' + col_name + ' dtype:' + dtypes + ' samples:' + sample_values

# Load training metadata
try:
    meta_df = train_metadata_df.copy()
except NameError:
    meta_df = pd.read_csv('your_training_data.csv')

meta_df = meta_df.fillna('')

canonical_map = {
    'ID': 'Customer ID',
    'Date': 'Date',
    'Numeric': 'Total Amount',
    'Categorical': 'Product Category',
    'Location': 'Region',
    'Text': 'Product Category',
    'Unknown': 'Product Category'
}

meta_df['backend_label'] = meta_df['label'].map(
    lambda x: canonical_map.get(str(x), 'Product Category')
)

X_text = build_feature_text(meta_df[['column_name', 'sample_values', 'dtype']])
y_raw = meta_df['backend_label'].astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

stratify_y = y if pd.Series(y).value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=stratify_y
)

model_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

print(f'Training rows: {len(meta_df)}')
print(f'Backend classes: {list(label_encoder.classes_)}')
model_pipeline.fit(X_train, y_train)

test_acc = model_pipeline.score(X_test, y_test) if len(X_test) > 0 else 1.0
print(f'Validation accuracy: {test_acc:.4f}')

# Save required PKLs directly to Team5_module/models
models_dir = r'..\models'
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'column_mapper_model.pkl')
encoder_path = os.path.join(models_dir, 'column_mapper_label_encoder.pkl')

joblib.dump(model_pipeline, model_path)
joblib.dump(label_encoder, encoder_path)

print('\nSuccess! Backend-compatible PKLs generated:')
print(f' - {model_path}')
print(f' - {encoder_path}')

Training rows: 18
Backend classes: ['Customer ID', 'Date', 'Product Category', 'Region', 'Total Amount']
Validation accuracy: 0.0000

Success! Backend-compatible PKLs generated:
 - ..\models\column_mapper_model.pkl
 - ..\models\column_mapper_label_encoder.pkl
